# Lesson 7: Market Pulse - Window Functions

The director walks in and drops three questions on your desk:

1. **"Who are the top 3 agents in each neighborhood?"**
2. **"What's the running total of commission payments over the year?"**
3. **"How does each property rank against its neighborhood peers by price?"**

You reach for `groupBy()` - then stop. These questions cannot be answered with `groupBy` alone.  
Why? Because `groupBy` **collapses rows**. Once you group, the individual property rows are gone.

### Window Functions vs GroupBy

| | `groupBy` | Window Function |
|---|---|---|
| Output rows | One per group | Same as input |
| Individual row detail | Lost | Preserved |
| Use case | Summary stats | Ranking, running totals, comparisons |

A **window function** computes a value for each row based on a **window** of related rows,  
then attaches that value back to the original row. The row is never collapsed.

In this lesson you will learn:
- `Window.partitionBy()` and `orderBy()` to define the window frame
- `row_number()`, `rank()`, `dense_rank()` for ranking
- `ntile()` for quartiles and percentile buckets
- `lag()` and `lead()` for time-series comparisons
- Running totals and moving averages with windowed `sum()` and `avg()`


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, round, avg, sum, count,
    row_number, rank, dense_rank, ntile,
    lag, lead
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Lesson7_MarketPulse") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Load commissions
commissions = spark.read \
    .option("multiline", "true") \
    .json("data/agent_commissions.json")

# Load listings
listings = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("data/property_listings.csv") \
    .dropna(subset=["list_price", "neighborhood"])

# Load mortgage rates
mortgage_rates = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("data/mortgage_rates.csv")

print(f"Commissions : {commissions.count()} rows")
print(f"Listings    : {listings.count()} rows")
print(f"Mortgage rates: {mortgage_rates.count()} rows")

Commissions : 363 rows


Listings    : 458 rows
Mortgage rates: 731 rows


## Window Spec - Defining the Frame

Before using any window function you must define a **WindowSpec** - the specification  
that tells Spark which rows belong to the same window and in what order to process them.

```python
from pyspark.sql.window import Window

w = Window.partitionBy("group_col").orderBy(col("sort_col").desc())
```

- **`partitionBy`** - like `groupBy`, it splits the data into independent groups.  
  Ranking resets to 1 at the start of each partition.
- **`orderBy`** - within each partition, defines the sequence (who is first, second, etc.).

A WindowSpec without `partitionBy` treats the **entire DataFrame** as one window.


In [2]:
# Window: rank properties within each neighborhood by list_price descending
# partitionBy = "reset the counter for each neighborhood"
# orderBy desc = "most expensive property gets rank 1"
window_by_neighborhood = Window \
    .partitionBy("neighborhood") \
    .orderBy(col("list_price").desc())

print("WindowSpec defined: partitioned by neighborhood, ordered by list_price descending")
print("\nThis window will:")
print("  - Treat each neighborhood as an independent group")
print("  - Rank properties from most to least expensive within each group")
print("  - Keep ALL original rows in the output (no collapsing!)")

# Preview the listings we will be ranking
listings.select("property_id", "address", "neighborhood", "list_price", "property_type") \
    .orderBy("neighborhood", col("list_price").desc()) \
    .show(8)

WindowSpec defined: partitioned by neighborhood, ordered by list_price descending

This window will:
  - Treat each neighborhood as an independent group
  - Rank properties from most to least expensive within each group
  - Keep ALL original rows in the output (no collapsing!)


+-----------+--------------+------------+----------+-------------+
|property_id|       address|neighborhood|list_price|property_type|
+-----------+--------------+------------+----------+-------------+
|  PROP-0306| 7214 River Rd|    Downtown| 1496853.0|        House|
|  PROP-0467|  5242 Main St|    Downtown| 1480728.0|        Condo|
|  PROP-0365|  5549 Main St|    Downtown| 1454295.0|        Condo|
|  PROP-0377|   771 Oak Ave|    Downtown| 1411368.0|    Townhouse|
|  PROP-0227|  4480 Hill Dr|    Downtown| 1350847.0|        House|
|  PROP-0108| 3011 River Rd|    Downtown| 1315123.0|        House|
|  PROP-0217|9187 Park Blvd|    Downtown| 1283153.0|        Condo|
|  PROP-0349| 1177 River Rd|    Downtown| 1279004.0|        Condo|
+-----------+--------------+------------+----------+-------------+
only showing top 8 rows


## Row Number - Unique Ranking

`row_number()` assigns a unique sequential integer to each row within its partition.  
No ties - even if two properties have the same price, they get different row numbers.

This is the tool for **"top N per group"** queries:  
rank every property within its neighborhood, then filter to keep only the top 3.


In [3]:
# Add row_number to each property, ranked by price within its neighborhood
listings_ranked = listings.withColumn(
    "price_rank",
    row_number().over(window_by_neighborhood)
)

print("--- All Properties with Price Rank (sample) ---")
listings_ranked.select(
    "neighborhood", "property_id", "address", "list_price", "price_rank"
).orderBy("neighborhood", "price_rank").show(12)

# Top 3 most expensive properties per neighborhood
print("--- Top 3 Most Expensive Properties Per Neighborhood ---")
listings_ranked.filter(col("price_rank") <= 3) \
    .select(
        "neighborhood", "price_rank", "address",
        "property_type", "list_price", "bedrooms"
    ) \
    .orderBy("neighborhood", "price_rank") \
    .show(30)

--- All Properties with Price Rank (sample) ---


+------------+-----------+--------------+----------+----------+
|neighborhood|property_id|       address|list_price|price_rank|
+------------+-----------+--------------+----------+----------+
|    Downtown|  PROP-0306| 7214 River Rd| 1496853.0|         1|
|    Downtown|  PROP-0467|  5242 Main St| 1480728.0|         2|
|    Downtown|  PROP-0365|  5549 Main St| 1454295.0|         3|
|    Downtown|  PROP-0377|   771 Oak Ave| 1411368.0|         4|
|    Downtown|  PROP-0227|  4480 Hill Dr| 1350847.0|         5|
|    Downtown|  PROP-0108| 3011 River Rd| 1315123.0|         6|
|    Downtown|  PROP-0217|9187 Park Blvd| 1283153.0|         7|
|    Downtown|  PROP-0349| 1177 River Rd| 1279004.0|         8|
|    Downtown|  PROP-0314|  5064 Main St| 1264100.0|         9|
|    Downtown|  PROP-0474| 5667 River Rd| 1210092.0|        10|
|    Downtown|  PROP-0397|1746 Park Blvd| 1193361.0|        11|
|    Downtown|  PROP-0162|7906 Park Blvd| 1192092.0|        12|
+------------+-----------+--------------

+------------+----------+--------------+-------------+----------+--------+
|neighborhood|price_rank|       address|property_type|list_price|bedrooms|
+------------+----------+--------------+-------------+----------+--------+
|    Downtown|         1| 7214 River Rd|        House| 1496853.0|       4|
|    Downtown|         2|  5242 Main St|        Condo| 1480728.0|       4|
|    Downtown|         3|  5549 Main St|        Condo| 1454295.0|       4|
|  Greenfield|         1|  5434 Hill Dr|        House| 1497203.0|       2|
|  Greenfield|         2|3172 Park Blvd|        Condo| 1488563.0|       2|
|  Greenfield|         3|  7883 Oak Ave|        Condo| 1479785.0|       2|
|    Hillside|         1| 8892 River Rd|    Townhouse| 1489527.0|       4|
|    Hillside|         2|  4318 Main St|        Condo| 1477148.0|       4|
|    Hillside|         3|  9211 Oak Ave|    Townhouse| 1454309.0|       4|
|     Midtown|         1|  4568 Oak Ave|        House| 1497758.0|       2|
|     Midtown|         2|

## Rank vs Dense Rank - Handling Ties

When two agents earn the same commission, what rank do they each get?

- **`rank()`** - tied rows get the same rank, then the next rank **skips**. (1, 1, 3, 4...)
- **`dense_rank()`** - tied rows get the same rank, next rank does **not** skip. (1, 1, 2, 3...)
- **`row_number()`** - no ties, always sequential. (1, 2, 3, 4...)

Which to use? Use `dense_rank` for "top N" filters when you want to include all tied entries.  
Use `row_number` when you need exactly N results regardless of ties.


In [4]:
# Rank agents by commission amount (ties are possible)
window_agents = Window.partitionBy("agent_id").orderBy(col("commission_amount").desc())

# Global ranking across all agents by single commission amount
window_global = Window.orderBy(col("commission_amount").desc())

commissions_ranked = commissions.withColumn(
    "rn", row_number().over(window_global)
).withColumn(
    "rnk", rank().over(window_global)
).withColumn(
    "dense_rnk", dense_rank().over(window_global)
)

print("--- Comparing row_number vs rank vs dense_rank ---")
print("Look for tied commission_amount values to see the difference:\n")
commissions_ranked.select(
    "agent_id", "commission_amount", "rn", "rnk", "dense_rnk"
).orderBy(col("commission_amount").desc()).show(20)

print("Key differences:")
print("  row_number : always unique, no ties")
print("  rank       : tied values share a rank; next rank skips")
print("  dense_rank : tied values share a rank; next rank does NOT skip")

--- Comparing row_number vs rank vs dense_rank ---
Look for tied commission_amount values to see the difference:



+--------+-----------------+---+---+---------+
|agent_id|commission_amount| rn|rnk|dense_rnk|
+--------+-----------------+---+---+---------+
| AGT-009|         41743.91|  1|  1|        1|
| AGT-002|         41668.69|  2|  2|        2|
| AGT-005|         41378.26|  3|  3|        3|
| AGT-015|         41320.69|  4|  4|        4|
| AGT-005|         40569.62|  5|  5|        5|
| AGT-009|         40070.59|  6|  6|        6|
| AGT-005|         39933.29|  7|  7|        7|
| AGT-006|         39429.95|  8|  8|        8|
| AGT-018|          39018.8|  9|  9|        9|
| AGT-018|         38336.13| 10| 10|       10|
| AGT-009|          37913.3| 11| 11|       11|
| AGT-008|          37836.3| 12| 12|       12|
| AGT-009|         37485.38| 13| 13|       13|
| AGT-002|         37454.87| 14| 14|       14|
| AGT-014|         37280.04| 15| 15|       15|
| AGT-012|         37250.85| 16| 16|       16|
| AGT-009|         37054.01| 17| 17|       17|
| AGT-011|         36806.04| 18| 18|       18|
| AGT-004|   

## Quartiles with ntile()

`ntile(n)` divides the rows within a partition into **n roughly equal buckets**  
and assigns each row a bucket number from 1 to n.

With `ntile(4)` you get **quartiles**: Q1 (bottom 25%), Q2, Q3, Q4 (top 25%).

This is extremely useful for reporting: "which price quartile is each property in,  
relative to its own neighborhood?"


In [5]:
# Assign price quartile within each neighborhood
window_for_quartile = Window.partitionBy("neighborhood").orderBy("list_price")

listings_with_quartile = listings.withColumn(
    "price_quartile",
    ntile(4).over(window_for_quartile)
)

print("--- Properties with Price Quartile (within neighborhood) ---")
listings_with_quartile.select(
    "neighborhood", "address", "list_price", "property_type", "price_quartile"
).orderBy("neighborhood", "list_price").show(12)

# Distribution of quartiles across neighborhoods
print("--- Quartile Distribution by Property Type ---")
listings_with_quartile.groupBy("property_type", "price_quartile") \
    .count() \
    .orderBy("property_type", "price_quartile") \
    .show(20)

--- Properties with Price Quartile (within neighborhood) ---


+------------+--------------+----------+-------------+--------------+
|neighborhood|       address|list_price|property_type|price_quartile|
+------------+--------------+----------+-------------+--------------+
|    Downtown|  1056 Main St|  196250.0|    Townhouse|             1|
|    Downtown|2998 Park Blvd|  202949.0|        House|             1|
|    Downtown|  817 River Rd|  213965.0|    Apartment|             1|
|    Downtown|  9862 Main St|  216253.0|         NULL|             1|
|    Downtown| 5832 River Rd|  238422.0|    Apartment|             1|
|    Downtown|  6919 Oak Ave|  277572.0|        House|             1|
|    Downtown|5641 Park Blvd|  289005.0|        Villa|             1|
|    Downtown|  6051 Hill Dr|  374737.0|        Villa|             1|
|    Downtown|  3031 Oak Ave|  418508.0|    Apartment|             1|
|    Downtown|  6646 Oak Ave|  437635.0|    Townhouse|             1|
|    Downtown|  3014 Main St|  468953.0|    Townhouse|             1|
|    Downtown|  8967

+-------------+--------------+-----+
|property_type|price_quartile|count|
+-------------+--------------+-----+
|         NULL|             1|    7|
|         NULL|             2|   10|
|         NULL|             3|    7|
|         NULL|             4|    4|
|    Apartment|             1|   25|
|    Apartment|             2|   17|
|    Apartment|             3|   18|
|    Apartment|             4|   12|
|        Condo|             1|   30|
|        Condo|             2|   27|
|        Condo|             3|   21|
|        Condo|             4|   36|
|        House|             1|   25|
|        House|             2|   37|
|        House|             3|   35|
|        House|             4|   34|
|    Townhouse|             1|   27|
|    Townhouse|             2|   22|
|    Townhouse|             3|   24|
|    Townhouse|             4|   24|
+-------------+--------------+-----+
only showing top 20 rows


## Lag and Lead - Looking at Neighbors in Time

`lag(col, n)` returns the value from **n rows before** the current row (within the partition).  
`lead(col, n)` returns the value from **n rows after** the current row.

These are the time-series tools. Sorted by date, `lag(1)` gives you yesterday's value -  
perfect for computing daily changes, rate movements, or period-over-period comparisons.

Applied to mortgage rates: what was yesterday's 30-year fixed rate, and how much did it change?


In [6]:
# Mortgage rate time series: compute daily change using lag()
window_by_date = Window.orderBy("date")

rates_with_change = mortgage_rates.withColumn(
    "prev_rate_30yr",
    lag("rate_30yr_fixed", 1).over(window_by_date)
).withColumn(
    "daily_change",
    round(
        col("rate_30yr_fixed") - col("prev_rate_30yr"),
        4
    )
).withColumn(
    "next_rate_30yr",
    lead("rate_30yr_fixed", 1).over(window_by_date)
)

print("--- Mortgage Rate Daily Changes (30yr Fixed) ---")
print("Note: first row has null prev_rate (no previous day)")
print("      last row has null next_rate (no next day)\n")
rates_with_change.select(
    "date", "rate_30yr_fixed", "prev_rate_30yr", "daily_change", "next_rate_30yr"
).show(10)

# Find the biggest single-day rate jumps
print("--- Top 5 Biggest Single-Day Rate Increases ---")
rates_with_change.filter(
    col("daily_change").isNotNull()
).orderBy(col("daily_change").desc()).select(
    "date", "rate_30yr_fixed", "prev_rate_30yr", "daily_change"
).show(5)

--- Mortgage Rate Daily Changes (30yr Fixed) ---
Note: first row has null prev_rate (no previous day)
      last row has null next_rate (no next day)



+----------+---------------+--------------+------------+--------------+
|      date|rate_30yr_fixed|prev_rate_30yr|daily_change|next_rate_30yr|
+----------+---------------+--------------+------------+--------------+
|2023-01-01|          6.711|          NULL|        NULL|         6.756|
|2023-01-02|          6.756|         6.711|       0.045|          6.81|
|2023-01-03|           6.81|         6.756|       0.054|         6.768|
|2023-01-04|          6.768|          6.81|      -0.042|         6.467|
|2023-01-05|          6.467|         6.768|      -0.301|         6.765|
|2023-01-06|          6.765|         6.467|       0.298|         6.661|
|2023-01-07|          6.661|         6.765|      -0.104|         6.895|
|2023-01-08|          6.895|         6.661|       0.234|         6.849|
|2023-01-09|          6.849|         6.895|      -0.046|         6.093|
|2023-01-10|          6.093|         6.849|      -0.756|         6.435|
+----------+---------------+--------------+------------+--------

+----------+---------------+--------------+------------+
|      date|rate_30yr_fixed|prev_rate_30yr|daily_change|
+----------+---------------+--------------+------------+
|2023-12-13|          6.907|         5.934|       0.973|
|2024-08-28|          6.709|          5.77|       0.939|
|2023-12-31|          6.972|         6.064|       0.908|
|2023-09-04|          6.709|         5.808|       0.901|
|2024-03-07|          7.233|         6.334|       0.899|
+----------+---------------+--------------+------------+
only showing top 5 rows


## Running Totals and Moving Averages

Window functions can also apply aggregate functions like `sum()` and `avg()`  
over a sliding frame - enabling running totals and moving averages.

The key is the **frame specification** added to the WindowSpec:

```python
# Running total: from the first row up to and including current row
Window.partitionBy(...).orderBy(...).rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Moving average (last 7 rows including current):
Window.orderBy(...).rowsBetween(-6, 0)
```


In [7]:
# Running total of commission amounts per agent, ordered by sale_date
window_agent_running = Window \
    .partitionBy("agent_id") \
    .orderBy("sale_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

commissions_running_total = commissions.withColumn(
    "running_total_commission",
    round(sum("commission_amount").over(window_agent_running), 2)
)

print("--- Running Commission Total per Agent (sample agent) ---")
# Show one agent's progression
sample_agent = commissions.orderBy(
    col("commission_amount").desc()
).select("agent_id").first()["agent_id"]

commissions_running_total.filter(col("agent_id") == sample_agent) \
    .select("agent_id", "sale_date", "commission_amount", "running_total_commission") \
    .orderBy("sale_date") \
    .show(10)

# 7-day moving average of 30yr mortgage rate
window_7day = Window.orderBy("date").rowsBetween(-6, 0)

rates_moving_avg = mortgage_rates.withColumn(
    "moving_avg_7day",
    round(avg("rate_30yr_fixed").over(window_7day), 4)
)

print("--- 7-Day Moving Average of 30yr Mortgage Rate ---")
rates_moving_avg.select(
    "date", "rate_30yr_fixed", "moving_avg_7day"
).orderBy("date").show(15)

--- Running Commission Total per Agent (sample agent) ---


+--------+----------+-----------------+------------------------+
|agent_id| sale_date|commission_amount|running_total_commission|
+--------+----------+-----------------+------------------------+
| AGT-009|2025-10-18|          6016.08|                 6016.08|
| AGT-009|2025-10-18|         17970.66|                23986.74|
| AGT-009|2025-10-22|         19934.94|                43921.68|
| AGT-009|2025-10-23|         20071.62|                 63993.3|
| AGT-009|2025-10-25|          7675.22|                71668.52|
| AGT-009|2025-10-31|         41743.91|               113412.43|
| AGT-009|2025-11-03|         20835.27|                134247.7|
| AGT-009|2025-11-11|         12410.92|               146658.62|
| AGT-009|2025-11-12|         30470.91|               177129.53|
| AGT-009|2025-11-17|         33831.72|               210961.25|
+--------+----------+-----------------+------------------------+
only showing top 10 rows
--- 7-Day Moving Average of 30yr Mortgage Rate ---


+----------+---------------+---------------+
|      date|rate_30yr_fixed|moving_avg_7day|
+----------+---------------+---------------+
|2023-01-01|          6.711|          6.711|
|2023-01-02|          6.756|         6.7335|
|2023-01-03|           6.81|          6.759|
|2023-01-04|          6.768|         6.7613|
|2023-01-05|          6.467|         6.7024|
|2023-01-06|          6.765|         6.7128|
|2023-01-07|          6.661|         6.7054|
|2023-01-08|          6.895|         6.7317|
|2023-01-09|          6.849|          6.745|
|2023-01-10|          6.093|         6.6426|
|2023-01-11|          6.435|          6.595|
|2023-01-12|          6.096|          6.542|
|2023-01-13|          6.169|         6.4569|
|2023-01-14|          6.673|         6.4586|
|2023-01-15|           6.95|         6.4664|
+----------+---------------+---------------+
only showing top 15 rows


In [8]:
# Full neighborhood leaderboard:
# Rank neighborhoods by avg price, show rank + total listings + avg price side by side

# Step 1: aggregate to get per-neighborhood stats
neighborhood_summary = listings.groupBy("neighborhood").agg(
    count("property_id").alias("total_listings"),
    round(avg("list_price"), 0).alias("avg_list_price"),
    round(avg("sqft"), 0).alias("avg_sqft")
)

# Step 2: define a window over the aggregated result
window_leaderboard = Window.orderBy(col("avg_list_price").desc())

# Step 3: add rankings
neighborhood_leaderboard = neighborhood_summary \
    .withColumn("price_rank", rank().over(window_leaderboard)) \
    .withColumn("listings_rank",
        rank().over(Window.orderBy(col("total_listings").desc()))
    ) \
    .orderBy("price_rank")

print("=" * 65)
print("        NEIGHBORHOOD LEADERBOARD - MARKET PULSE REPORT")
print("=" * 65)
neighborhood_leaderboard.select(
    "price_rank",
    "neighborhood",
    "avg_list_price",
    "total_listings",
    "listings_rank",
    "avg_sqft"
).show(truncate=False)

        NEIGHBORHOOD LEADERBOARD - MARKET PULSE REPORT


+----------+------------+--------------+--------------+-------------+--------+
|price_rank|neighborhood|avg_list_price|total_listings|listings_rank|avg_sqft|
+----------+------------+--------------+--------------+-------------+--------+
|1         |Riverside   |914969.0      |61            |2            |2896.0  |
|2         |Midtown     |841423.0      |48            |8            |2813.0  |
|3         |Downtown    |840034.0      |70            |1            |3125.0  |
|4         |Oakwood     |838114.0      |49            |6            |2752.0  |
|5         |Seaside     |836389.0      |60            |4            |2923.0  |
|6         |Parkview    |825007.0      |49            |6            |2701.0  |
|7         |Hillside    |787136.0      |61            |2            |3082.0  |
|8         |Greenfield  |765911.0      |60            |4            |3079.0  |
+----------+------------+--------------+--------------+-------------+--------+



## Lesson 7 Wrap-Up

The director has their answers. You produced rankings, running totals, and market comparisons  
without losing a single row of detail.

**What you learned:**
- `Window.partitionBy().orderBy()` - defines the window frame
- `row_number()` - unique sequential rank, no ties
- `rank()` - ties share a rank; next rank skips
- `dense_rank()` - ties share a rank; next rank does NOT skip
- `ntile(n)` - assigns rows to n equally-sized buckets (quartiles, deciles, etc.)
- `lag(col, n)` - value from n rows before current row
- `lead(col, n)` - value from n rows after current row
- `sum().over(window)` with `rowsBetween` - running totals
- `avg().over(window)` with `rowsBetween(-n, 0)` - moving averages

**The golden rule of window functions:**  
Use `groupBy` when you want a summary. Use window functions when you need the summary  
**attached back to each individual row**.

---

**Next: Lesson 8 - Enriching the Listings**  
Learn how to transform, clean, and enrich raw data with derived columns,  
string operations, date parsing, and conditional logic using `when()` / `otherwise()`.
